In [2]:
import pandas as pd
import numpy as np

#LOAD DATA

df = pd.read_csv('e_waste_dataset_with_profit.csv')
print(" Dataset loaded:", df.shape)

# EXPLORE DATASET  

print("\n Columns:", df.columns.tolist())
print("\n First 5 rows:")
print(df.head())
print("\n Summary statistics:")
print(df.describe())
print("\n Data types:")
print(df.dtypes)
print("\n Missing values:")
print(df.isnull().sum())

# HANDLE MISSING VALUES

if df.isnull().sum().sum() > 0:
    print("\n🧹 Filling missing values with median...")
    df = df.fillna(df.median())
    print(" Missing values handled")

# REMOVE OUTLIERS (IQR method)

def remove_outliers(df, columns):
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean

metal_cols = ['Gold','Silver','Platinum','Rhodium','Nickel','Tin','Lithium','Aluminum','Carbon']
df = remove_outliers(df, metal_cols)
print(f"\n Outliers removed: {len(df)} rows remaining")

# CALCULATE TOTAL VALUE & RECOVERABLE COLUMN

# Industry standard 2026 scrap prices ($/gram)
prices_per_gram = {
    'Gold': 62, 'Platinum': 81, 'Silver': 0.95, 'Rhodium': 155,
    'Nickel': 0.018, 'Tin': 0.033, 'Lithium': 0.012, 
    'Aluminum': 0.003, 'Carbon': 0.0001
}

# Calculate total value for each row
df['Total_Value'] = 0.0
for metal, price in prices_per_gram.items():
    df['Total_Value'] += df[metal] * price

# Threshold: Top 35% most valuable = recoverable (industry realistic)
threshold = df['Total_Value'].quantile(0.65)  # 35% recoverable
df['Recoverable'] = (df['Total_Value'] > threshold).astype(int)

# Move Recoverable to rightmost column
cols = list(df.columns)
cols.remove('Recoverable')
cols.append('Recoverable')
df = df[cols]

print(f"\n Threshold: ${threshold:.2f}")
print(" Recoverable distribution:")
print(df['Recoverable'].value_counts(normalize=True))
print("\n Final dataset:")
print(df[['Total_Value', 'Recoverable']].describe())
print("\n Dataset ready! Shape:", df.shape)

# Save cleaned dataset
df.to_csv('e_waste_cleaned_recoverable.csv', index=False)
print("\n Saved: e_waste_cleaned_recoverable.csv")


 Dataset loaded: (10000, 12)

 Columns: ['Item', 'Category', 'Gold', 'Silver', 'Platinum', 'Rhodium', 'Nickel', 'Tin', 'Lithium', 'Aluminum', 'Carbon', 'Profit ($)']

 First 5 rows:
              Item Category  Gold  Silver  Platinum  Rhodium  Nickel   Tin  \
0        iPhone 11     Cat3  3.58    2.95      1.73     8.92    1.91  1.01   
1          Toaster     Cat2  7.21    4.31      6.21     5.63    9.59  7.65   
2          Speaker     Cat4  8.91    5.09      2.42     7.70    1.09  1.49   
3   Microwave Oven     Cat2  2.62    3.84      2.98     7.66    9.41  2.25   
4  Air Conditioner     Cat1  3.47    3.89      6.20     4.35    5.07  8.65   

   Lithium  Aluminum  Carbon  Profit ($)  
0     1.82      1.27    9.51      270.34  
1     0.51      3.03    4.22      689.75  
2     7.42      3.63    8.83      570.43  
3     7.84      6.18    6.36      290.78  
4     8.62      0.82    5.53      505.16  

 Summary statistics:
               Gold        Silver      Platinum       Rhodium        